# BoardGameGeek Data Pipeline: Engineering & Analysis Report
**Author:** Adam Jaworski

## 1. Project Overview
The objective of this project was to build a robust data extraction pipeline to compile a dataset of board game metadata from BoardGameGeek (BGG) and enrich it with historical award data (Spiel des Jahres). 

Because modern web platforms employ aggressive anti-bot protections and dynamic JavaScript rendering, standard synchronous HTTP requests (`requests` library) were insufficient for the primary target. This necessitated the development of a hybrid scraping architecture to ensure meeting the requirements.

## 2. Step by step

To successfully aggregate this dataset without violating server constraints or triggering automated bans, the data engineering pipeline was divided into four distinct phases, moving from raw acquisition to final integration.

* **Phase 1: Target Acquisition (Selenium & BeautifulSoup)**
    The process began by identifying the top 1,000 board games ranked on BoardGameGeek. Because the site utilizes dynamic consent popups (GDPR/Cookies) that block raw HTML parsing, Selenium WebDriver was deployed to programmatically close these overlays, extract the base URLs for each game, and save them to a localized `raw_urls.csv` file.
* **Phase 2: Deep Metadata Extraction (Scrapy & Selenium)**
    With the target list secured, the core extraction phase was launched using the Scrapy framework. To combat Cloudflare Bot Management and Angular JavaScript rendering delays, a custom `SeleniumMiddleware` was built. This hybrid approach allowed the Scrapy engine to manage the asynchronous request queue while routing the actual page-loads through a headless Chrome browser, successfully capturing intricate metadata (playtime, weight, designers) into `metadata.csv`.
* **Phase 3: Award Enrichment (Requests & BeautifulSoup)**
    To fulfill the requirement of utilizing standard HTTP requests without relying on APIs, the pipeline pivoted to Wikipedia to extract the historical list of *Spiel des Jahres* (Game of the Year) winners. The result is `awards.csv` file. 
* **Phase 4: Data Integration & Entity Resolution (Pandas)**
    The final phase involved synthesizing the three disparate CSV files. By normalizing all titles (stripping punctuation, standardizing casing), the BGG metadata and Wikipedia award history were merged into the `final_dataset.csv` without dropping unawarded baseline games.

## 3. Technical Challenges & Engineered Solutions

During the development of this pipeline, several severe infrastructural and data-quality roadblocks were encountered. Below is a summary of the issues and the engineered workarounds:

### Challenge 1: Cloudflare WAF (Web Application Firewall) Blocking
* **The Issue:** Any `requests` or Scrapy spiders were instantly blocked with HTTP 403 (Forbidden) errors.
* **The Solution:** `SeleniumMiddleware` for Scrapy. Scrapy managed the asynchronous queue, while Selenium handled the actual network request in a Chrome browser.

### Challenge 2: Angular Framework DOM Delays
* **The Issue:** Even after bypassing Cloudflare, extracted fields were returning empty. BGG's Angular frontend took an additional 1-2 seconds to inject data into the DOM after the page loaded.
* **The Solution:** Force Selenium to wait for things to physically render on the screen before capturing the page source.

### Challenge 3: Browser Memory Leaks & Session Bans
* **The Issue:** Scraping 1,000 pages consecutively caused the browser to gradualy slow down and triggered session-length bans, freezing the scraper at ~400 records.
* **The Solution:** Engineered a reset mechanism in the middleware. Every 75 requests, the script explicitly killed the browser, flushed the RAM, and generated a fresh network footprint. Additionally, a strict 30-second timeout was added; if a page hung, the middleware returned an HTTP 500 status, triggering Scrapy's built-in Retry Engine to rescue the dropped URL later.

### Challenge 4: Entity Resolution
* **The Issue:** Merging the Wikipedia dataset with the BGG dataset failed initially because Wikipedia included citations (`[1]`) and German translations in parentheses alongside the English titles.
* **The Solution:** Deployed Regex cleaning during extraction and built a Pandas `normalize_title()` function to strip punctuation, remove extraneous spaces, and convert strings to lowercase before executing the Left Join.

## 4. Verification of Merge Integrity
To verify that our normalization logic successfully matched games between BGG and Wikipedia, we can filter the dataset to display only the games that successfully received the 'Won' status from the secondary scrape.

In [2]:
import pandas as pd

# 1. Load the Datasets
final_df = pd.read_csv('data/final_dataset.csv', dtype={'Year_Won': 'Int64'})

# Drop rows where the Award_Status is literally 'None'
winners = final_df[final_df['Spiel_des_Jahres'] == True].copy()

print(f"Successfully matched Spiel des Jahres winners in the Top 1000: {len(winners)}")
winners[['Rank', 'Title', 'Year', 'Year_Won', 'Spiel_des_Jahres']].head(15)

Successfully matched Spiel des Jahres winners in the Top 1000: 21


,Rank,Title,Year,Year_Won,Spiel_des_Jahres
32,33,Sky Team,2023,2024,True
59,60,Cascadia,2021,2022,True
95,96,Azul,2017,2018,True
99,100,El Grande,1995,1996,True
100,101,Bomb Busters,2024,2025,True
144,145,Dominion,2008,2009,True
156,157,Just One,2018,2019,True
160,161,Codenames,2015,2016,True
238,239,Carcassonne,2000,2001,True
255,256,Ticket to Ride,2004,2004,True
